In [2]:
import numpy as np

In [10]:

import jieba
import gensim

from gensim import corpora, models
from gensim import similarities


class TFIDFTextMatch:
    def __init__(self):
        self.documents = []
        self.model = None
        self.dictionary = None
        self.corpus = None
    
    def build(self):
        self.build_dictionary()
        self.build_corpus()
        self.create_model()
        self.create_index()

    def set_documents(self, documents, cutted=False):
        if cutted:
            self.documents = documents
        else:
            self.documents = [[word for word in list(jieba.cut(document))] for document in documents]
    
    def build_dictionary(self):
        self.dictionary = corpora.Dictionary(self.documents)
    
    def build_corpus(self):
        self.corpus = [self.dictionary.doc2bow(doc) for doc in self.documents]
    
    def create_model(self):
        self.model = models.TfidfModel(self.corpus)
    
    def create_index(self):
        self.index = similarities.SparseMatrixSimilarity(self.model[self.corpus], num_features=len(self.dictionary))

    def query(self, query_document, cutted=False):
        if cutted and isinstance(query_document, str):
            query_document = [query_document]
        if not cutted:
            query_document = list(jieba.cut(query_document))
        query_bow = self.dictionary.doc2bow(query_document)
        similarities  = self.index[self.model[query_bow]]
        return list(enumerate(similarities))
    
    def search(self, query_document, top_k = 5, cutted=False):
        results = []
        similarities = self.query(query_document, cutted)
        for document_number, score in sorted(similarities, key=lambda x: x[1], reverse=True)[:top_k]:
            results.append((document_number, ''.join(self.documents[document_number]), score))
        return results
            



In [11]:
documents = [
    '目标检测',
    '文本匹配',
    '物体检测',
    '图像识别',
    '图像分类',
    '目标跟踪',
    '图像分割',
]

searcher = TFIDFTextMatch()
searcher.set_documents(documents)
searcher.build()

In [12]:
searcher.search('目标分类', cutted=False)

[(4, '图像分类', 0.706979),
 (0, '目标检测', 0.38276687),
 (5, '目标跟踪', 0.293021),
 (1, '文本匹配', 0.0),
 (2, '物体检测', 0.0)]

In [8]:
searcher.search(['目标','分类'], cutted=True)

[(4, '图像分类', 0.706979),
 (0, '目标检测', 0.38276687),
 (5, '目标跟踪', 0.293021),
 (1, '文本匹配', 0.0),
 (2, '物体检测', 0.0)]

In [6]:
%%timeit
searcher.search(query, cutted=True)

129 µs ± 1.18 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
